## Notes

- The pipeline now uses a 1% validation split and a slower checkpoint/logging cadence to keep the training loop practical for a 30B model.
- For actual Nemotron-3-Nano-30B training, run this notebook on a CUDA machine with enough VRAM or adapt the loading block to use a 4-bit quantized setup.
- If the model's attention module names differ, update `candidate_targets` after inspecting `model.named_modules()`.
- If you want, the next step can be a fully quantized QLoRA variant or a TRL `SFTTrainer` version.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
)

if RUN_TRAINING:
    train_result = trainer.train()
    print(train_result)
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    model.save_pretrained(str(OUTPUT_DIR))
else:
    print("RUN_TRAINING is False. Review the pipeline and set it to True on a suitable GPU machine.")

# Smoke test the prompt formatting so the notebook still validates the inference path.
def generate_answer(prompt_text, max_new_tokens=64):
    model.eval()
    if USE_CHAT_TEMPLATE and getattr(tokenizer, "chat_template", None):
        messages = [{"role": "user", "content": prompt_text}]
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        input_text = f"### Instruction:\n{prompt_text}\n\n### Response:\n"

    inputs = tokenizer(input_text, return_tensors="pt")
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(generate_answer(train_df.iloc[0]["prompt"])[:800])

## 5. Train, Save, and Smoke Test

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    weight_decay=0.0,
    logging_steps=LOGGING_STEPS,
    evaluation_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    bf16=torch.cuda.is_available(),
    fp16=False,
    remove_unused_columns=False,
    report_to="none",
    optim="adamw_torch",
)

# Tokenize with a fixed length; this keeps the pipeline simple and reproducible.
def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

train_tokenized = train_text_ds.map(tokenize_batch, batched=True, remove_columns=train_text_ds.column_names)
eval_tokenized = eval_text_ds.map(tokenize_batch, batched=True, remove_columns=eval_text_ds.column_names)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

print(train_tokenized[0])


## 4. Training Configuration

In [ ]:
# For a real training run, this should be executed on a CUDA machine with enough VRAM.
# If you have quantization support available, you can extend this with 4-bit loading.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

candidate_targets = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
found_targets = sorted({name.split('.')[-1] for name, _ in model.named_modules() if name.split('.')[-1] in candidate_targets})
if not found_targets:
    found_targets = candidate_targets

print(f"LoRA target modules: {found_targets}")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=found_targets,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 3. Load Model and Configure LoRA

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
print(f"Pad token: {tokenizer.pad_token}")

if USE_CHAT_TEMPLATE and getattr(tokenizer, "chat_template", None):
    def format_example(example):
        messages = [
            {"role": "user", "content": example["prompt"]},
            {"role": "assistant", "content": normalize_answer(example["answer"])}
        ]
        return {"text": tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)}
else:
    def format_example(example):
        text = (
            f"### Instruction:\n{example['prompt']}\n\n"
            f"### Response:\n{normalize_answer(example['answer'])}{tokenizer.eos_token}"
        )
        return {"text": text}

train_text_ds = train_ds.map(format_example, remove_columns=train_ds.column_names)
eval_text_ds = eval_ds.map(format_example, remove_columns=eval_ds.column_names)

print(train_text_ds[0]["text"][:500])

In [ ]:
def normalize_answer(text):
    text = str(text).strip()
    text = unicodedata.normalize('NFKC', text)
    text = re.sub(r'\s+', ' ', text)
    return text

# Keep a small validation split for quick checks on the larger corpus.
train_split_df, eval_split_df = train_test_split(
    train_df[['prompt', 'answer']],
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    shuffle=True,
)

train_split_df = train_split_df.reset_index(drop=True)
eval_split_df = eval_split_df.reset_index(drop=True)

print(f"Train split: {train_split_df.shape}")
print(f"Eval split: {eval_split_df.shape}")

train_ds = Dataset.from_pandas(train_split_df)
eval_ds = Dataset.from_pandas(eval_split_df)


## 2. Prepare the Instruction Dataset

In [ ]:
import re
import unicodedata
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, TaskType, get_peft_model

# Reproducibility
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

train_df = pd.read_csv(TRAIN_PATH)
print(f"Loaded training data: {train_df.shape}")
print(train_df.head(3).to_string(index=False))

## 1. Load Libraries and Data

In [ ]:
# Core configuration
from pathlib import Path

PROJECT_DIR = Path(r"c:\Users\nduta\OneDrive\Desktop\Projects\nvidia-nemotron-nodel-reasoning-challenge")
TRAIN_PATH = PROJECT_DIR / "train.csv"
OUTPUT_DIR = PROJECT_DIR / "outputs" / "nemotron_lora_adapter"
MODEL_ID = "nvidia/Nemotron-3-Nano-30B"

# Training controls
MAX_LENGTH = 1024
TEST_SIZE = 0.01
RANDOM_STATE = 42
RUN_TRAINING = False  # Set to True on a GPU machine
USE_CHAT_TEMPLATE = True

# Conservative training defaults for a 30B model.
NUM_TRAIN_EPOCHS = 1
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 16
LEARNING_RATE = 1e-4
LOGGING_STEPS = 25
EVAL_STEPS = 500
SAVE_STEPS = 500

# LoRA defaults
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    data_collator=data_collator,
)

if RUN_TRAINING:
    train_result = trainer.train()
    print(train_result)
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    model.save_pretrained(str(OUTPUT_DIR))
else:
    print("RUN_TRAINING is False. Review the pipeline and set it to True on a suitable GPU machine.")

# Smoke test the prompt formatting so the notebook still validates the inference path.
def generate_answer(prompt_text, max_new_tokens=64):
    model.eval()
    if USE_CHAT_TEMPLATE and getattr(tokenizer, "chat_template", None):
        messages = [{"role": "user", "content": prompt_text}]
        input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        input_text = f"### Instruction:\n{prompt_text}\n\n### Response:\n"

    inputs = tokenizer(input_text, return_tensors="pt")
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
        )
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(generate_answer(train_df.iloc[0]["prompt"])[:800])

## 5. Train, Save, and Smoke Test

# Nemotron-3-Nano LoRA Training Pipeline

This notebook sets up a lightweight LoRA fine-tuning pipeline for the Nemotron-3-Nano-30B base model. It focuses on supervised instruction tuning from the provided `prompt` and `answer` columns and is structured to run on a CUDA-capable machine with enough memory for model loading or quantized training.